# Vector stores and semantic search



In [10]:
from sentence_transformers import SentenceTransformer
import torch
import csv

## Part I: Basic vector store implementation

In [12]:
class Document:
    def __init__(self, text: str, metadata: dict[str, str]):
        self.text = text
        self.metadata = metadata


class SearchResult:
    def __init__(self, score: float, document: Document):
        self.score = score
        self.document = document


class VectorStore:
    def __init__(self, embedding_model: SentenceTransformer):
        self.documents = []
        self.embeddings = None
        self.model = embedding_model

    def add_documents(self, documents: list[Document]):
        new_embeddings = self.model.encode(
            [doc.text for doc in documents],
            convert_to_tensor=True,
            normalize_embeddings=True,
        )

        if self.embeddings is None:
            self.embeddings = new_embeddings
        else:
            self.embeddings = torch.cat([self.embeddings, new_embeddings], dim=0)

        self.documents.extend(documents)

    def search(self, query: str, top_k: int = 5) -> list[SearchResult]:
        embeded_query = self.model.encode(query, convert_to_tensor=True, normalize_embeddings=True)
        scores = embeded_query @ self.embeddings.T

        sorted_scores, sorted_idx = torch.sort(scores, descending=True)
        top_scores = sorted_scores[:top_k]
        top_indices = sorted_idx[:top_k]

        return [SearchResult(score.item(), self.documents[idx]) for score, idx in zip(top_scores, top_indices)]

Dataset

In [13]:
documents = []

with open('data/animal-fun-facts-dataset.csv', mode='r', encoding='utf-8') as file:
    reader = csv.DictReader(file)
    for i, row in enumerate(reader):
        text = str(row.pop('text'))
        documents.append(Document(text=text, metadata=row))

print(f"Loaded {len(documents)} documents.\n")

for i, doc in enumerate(documents):
    if (i >= 5):
        break
    print(f"Document {i+1}:")
    print(doc.text)
    print(doc.metadata)
    print()

Loaded 7734 documents.

Document 1:
Aardvarks are sometimes called "ant bears", "earth pigs",
and "cape anteaters"
{'animal_name': 'aardvark', 'source': 'https://www.animalfactsencyclopedia.com/Aardvark-facts.html', 'media_link': '', 'wikipedia_link': '/wiki/Aardvark'}

Document 2:
Aardvarks
have rather primitive brains that are very small for the size of the
animal. Some have suggested they are not particularly bright....
{'animal_name': 'aardvark', 'source': 'https://www.animalfactsencyclopedia.com/Aardvark-facts.html', 'media_link': '', 'wikipedia_link': '/wiki/Aardvark'}

Document 3:
Aardvarks
teeth are lined with fine upright tubes and have no roots or enamel.
{'animal_name': 'aardvark', 'source': 'https://www.animalfactsencyclopedia.com/Aardvark-facts.html', 'media_link': '', 'wikipedia_link': '/wiki/Aardvark'}

Document 4:
The aardvarks Latin family name "Tubulidentata" means "tube toothed"
{'animal_name': 'aardvark', 'source': 'https://www.animalfactsencyclopedia.com/Aardvark-f

In [19]:
# Initialize the vector
model = SentenceTransformer('all-MiniLM-L6-v2')
vector_store = VectorStore(model)
vector_store.add_documents(documents)

# Queries
queries = [
    "Which animal can regenerate its limbs?",
    "How do whales communicate?",
    "What is the fastest land animal?",
    "Tell me a fun fact about elephants",
    "Which birds cannot fly?"
    ]

for query in queries:
    results = vector_store.search(query, top_k=5)

    for res in results:
        print(f"Score: {res.score:.4f}")
        print(f"Text: {res.document.text}")
        print(f"Metadata: {res.document.metadata}")
        print("\n")
    print("-" *50)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 13712.53it/s]


Score: 0.7367
Text: They can regenerate parts of their body! If they lose a limb it will grow back.
Metadata: {'animal_name': 'axolotl', 'source': '/r/AskReddit/comments/gbh7zz/what_are_some_really_amazing_animal_facts/fp67r9o/', 'media_link': '', 'wikipedia_link': '/wiki/Axolotl'}


Score: 0.7154
Text: Able to regrow lost or damaged limbs!
Metadata: {'animal_name': 'newt', 'source': 'https://a-z-animals.com/animals/newt/', 'media_link': '', 'wikipedia_link': '/wiki/Newt'}


Score: 0.6430
Text: They can regrow their arms.
This has been witnessed in one specimen from Newfoundland, in 1968, around the time when, for unknown reasons, these animals were washing up in droves.
Metadata: {'animal_name': 'giant squid', 'source': 'https://factanimal.com/giant-squid/', 'media_link': '', 'wikipedia_link': '/wiki/Giant_squid'}


Score: 0.6429
Text: Axolotl have an astonishing ability to regenerate body organs and lost limbs..
Incredibly, an Axolotl can grow back lost limbs in only a few weeks. It 

## Part II: Filtering by metadata

In [25]:
class FilteredVectorStore:
    def __init__(self, embedding_model: SentenceTransformer):
        self.documents = []
        self.embeddings = None
        self.model = embedding_model

    def add_documents(self, documents: list[Document]):
        new_embeddings = self.model.encode(
            [doc.text for doc in documents],
            convert_to_tensor=True,
            normalize_embeddings=True,
        )

        if self.embeddings is None:
            self.embeddings = new_embeddings
        else:
            self.embeddings = torch.cat([self.embeddings, new_embeddings], dim=0)

        self.documents.extend(documents)

    def search(self,
               query: str,
               top_k: int = 5,
               metadata_filter: dict[str, str] | None = None) -> list[SearchResult]:

        if not self.documents:
            return []

        embedded_query = self.model.encode([query], convert_to_tensor=True, normalize_embeddings=True)
        filtered_indices = list(range(len(self.documents)))

        if metadata_filter:
            candidate_indices = []
            for idx in filtered_indices:
                doc = self.documents[idx]
                filter_compliant = True
                for k, v in metadata_filter.items():
                    if k not in doc.metadata or doc.metadata[k] != v:
                        filter_compliant = False
                        break
                if filter_compliant:
                    candidate_indices.append(idx)

            filtered_indices = candidate_indices

            if not filtered_indices:
                return []

        filtered_embeddings = self.embeddings[filtered_indices]
        scores = embedded_query @ filtered_embeddings.T

        actual_k = min(top_k, len(filtered_indices))
        sorted_scores, sorted_idx = torch.sort(scores, descending=True)

        top_scores = sorted_scores[0, :actual_k]
        top_indices = sorted_idx[0, :actual_k]

        return [
            SearchResult(score.item(), self.documents[filtered_indices[idx]])
            for score, idx in zip(top_scores, top_indices)
        ]


In [31]:
# Initialize the vector
model = SentenceTransformer('all-MiniLM-L6-v2')
vector_store = FilteredVectorStore(model)
vector_store.add_documents(documents)

# Queries
print(vector_store.search("Armor", metadata_filter={"animal_name": "armadillo"}))

print ("\n")

print(vector_store.search("swim", metadata_filter={"animal_name": "turtle"}))

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9206.86it/s]


[Text: The armadillo's armor is made out of bone
Score: 0.5399112701416016, Text: Their names reflect their shells.
In Spanish, the armadillo’s name translates to “little armored one”.
Score: 0.34611979126930237, Text: Armadillo shells harden quickly.
Armadillo pup shells are soft when born, with a leather-like texture. Over the next couple of days, their shells will harden into protective armor similar to that of their mothers.
Score: 0.30733418464660645, Text: Their shells are made of bone.
Armadillo shells are composed of bony plates covered in scales. The part that covers their bodies is called a carapace and is made up of segmented bands.
Score: 0.2704770565032959, Text: Armadillos can swim underwater and hold their breath for 5 minutes
Score: 0.21715828776359558]


[Text: Some turtles can swim backwards. This one just saw a couple Manatees pass beneath it.
Score: 0.3799646496772766, Text: Turtles can breath through their butts.
Score: 0.24512003362178802, Text: Turtles can breath

**Dataset**

Dataset name is BBC News, here's the link to the dataset: https://www.kaggle.com/datasets/ervishvanathmetkari/bbc-news

In [35]:
documents = []

with open('data/BBC News Train.csv', mode='r', encoding='utf-8') as file:
    reader = csv.DictReader(file)
    for i, row in enumerate(reader):
        text = str(row.pop('Text'))
        documents.append(Document(text=text, metadata=row))

print(f"Loaded {len(documents)} documents.\n")

for i, doc in enumerate(documents):
    if (i >= 5):
        break
    print(f"Document {i+1}:")
    print(doc.text)
    print(doc.metadata)
    print()

Loaded 1490 documents.

Document 1:
worldcom ex-boss launches defence lawyers defending former worldcom chief bernie ebbers against a battery of fraud charges have called a company whistleblower as their first witness.  cynthia cooper  worldcom s ex-head of internal accounting  alerted directors to irregular accounting practices at the us telecoms giant in 2002. her warnings led to the collapse of the firm following the discovery of an $11bn (£5.7bn) accounting fraud. mr ebbers has pleaded not guilty to charges of fraud and conspiracy.  prosecution lawyers have argued that mr ebbers orchestrated a series of accounting tricks at worldcom  ordering employees to hide expenses and inflate revenues to meet wall street earnings estimates. but ms cooper  who now runs her own consulting business  told a jury in new york on wednesday that external auditors arthur andersen had approved worldcom s accounting in early 2001 and 2002. she said andersen had given a  green light  to the procedures and

In [45]:
# Initialize the vector
model = SentenceTransformer('all-MiniLM-L6-v2')
vector_store = FilteredVectorStore(model)
vector_store.add_documents(documents)

# Queries
print(vector_store.search("argentina", metadata_filter={"Category": "sport"}))
print("\n")
print(vector_store.search("microsoft", metadata_filter={"Category": "tech"}))
print("\n")
print(vector_store.search("oscar", metadata_filter={"Category": "entertainment"}))
print("\n")
print(vector_store.search("elections", metadata_filter={"Category": "politics"}))
print("\n")
print(vector_store.search("inflation", metadata_filter={"Category": "business"}))

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8188.27it/s]


[Text: ireland 21-19 argentina an injury-time dropped goal by ronan o gara stole victory for ireland from underneath the noses of argentina at lansdowne road on saturday.  o gara kicked all of ireland s points  with two dropped goals and five penalties  to give the home side a 100% record in their autumn internationals. an impressive argentina appeared in control until the dying seconds. the pumas shocked the irish early on with a try from federico aramburu  and felipe contepomi kicked 14 points. the well-drilled and sharper pumas out-played and out-thought ireland in the early stages. indiscipline allowed argentina s leinster fly-half contepomi to open the scoring in the third minute with a straightforward penalty. he was on the mark again two minutes later when argentina shocked a ragged ireland with the first try of the game. ireland turned the ball over and manuel contepomi broke through an unstructured defence before feeding his midfield partner aramburu to sprint in under the pos

**Final Reflections**

This activity helped me understand the concepts of vector store and semantic search by implementing a basic vector store and perform semantic search based on cosine similarity, also extended the implementation to allow filtering by metadata. It wasn't that easy as I expected at first, but, after some trial and error I was able to make it work.

Note: I used AI to help me with clean code and find and fix some minor errors.